# NTS §5.10.2.1 — Eigenvalue sweep (MGES only, no MPE)

Replicates the two-machine benchmark study from REE NTS 631 §5.10.2.1.
Line reactance XL is swept from 0.01 to 0.6 pu to visualise the evolution
of electromechanical modes on the complex plane.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from nts_emo import build_nts_model, xl_sweep, plot_eigenvalue_sweep
from nts_emo.assembly import assemble_system, get_system_info

## 1. Inspect the system

In [ ]:
sys_dict = assemble_system()
info = get_system_info(sys_dict)
print('System summary:')
for k, v in info.items():
    print(f'  {k}: {v}')
print('\nState variables:')
for i, x in enumerate(sys_dict['x_list']):
    print(f'  {i+1:2d}. {x}')

## 2. Build the compiled model (runs once, ~1-2 min)

In [ ]:
BUILD_DIR = './build'
build_nts_model(output_dir=BUILD_DIR)

## 3. Check initialisation at a single XL value

In [ ]:
from nts_emo.build_model import load_model
import pydae.ssa as ssa

model = load_model(BUILD_DIR)
ok = model.ini({'XL': 0.3}, xy_0=1.0)
print(f'Converged: {ok}')
print(f'  e_4 = {model.get_value("e_4"):.4f}  (expected 1.0)')
print(f'  f_4 = {model.get_value("f_4"):.6f} (expected 0.0)')
print(f'  Pe_1 = {model.get_value("Pe_1"):.4f} pu  (expected ~13.5)')

df = ssa.damp_report(model)

## 4. XL sweep

In [ ]:
xl_values = np.arange(0.01, 0.61, 0.05).round(3).tolist()
print(f'Sweeping {len(xl_values)} XL values: {xl_values}')

results = xl_sweep(xl_values, model_dir=BUILD_DIR)

print('\nElectromechanical modes:')
print(f'{"XL":>6}  {"sigma":>8}  {"freq (Hz)":>10}  {"damp (%)":>10}')
for xl, lam, f_hz, zeta in zip(
    results['xl_values'], results['em_modes'],
    results['em_freq_hz'], results['em_damp']
):
    if not np.isnan(f_hz):
        print(f'{xl:6.2f}  {lam.real:8.3f}  {f_hz:10.3f}  {zeta*100:10.2f}')

## 5. Complex-plane plot (replicating NTS Fig 17/18)

In [ ]:
fig = plot_eigenvalue_sweep(results)
plt.tight_layout()
plt.savefig('nts_eigenvalue_sweep.png', dpi=150)
plt.show()